In [1]:
# =====================================================================
# ETAPE 1 — Spark session + lecture silver
# =====================================================================
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("esg_greenwashing")
    .config("spark.hadoop.fs.s3a.endpoint", "minio.lab.sspcloud.fr")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.default.parallelism", "8")
    .enableHiveSupport()
    .getOrCreate()
)

print(f"Spark {spark.version} OK")

df = spark.read.parquet("s3a://sarangan/silver/esg_clean/")
print(f"Lignes : {df.count()}")
print(f"Colonnes : {len(df.columns)}")
df.show(3, truncate=False)

Spark 4.1.1 OK


Lignes : 450
Colonnes : 25


+-------------+------+-------+--------------+------------------------+------------------------+------------------------+-------------------+---------------------+-------------------------------+---------------+-----------------+-----------------------+------------------+-----------------------+------------------------+----------------+--------------------+------------------+--------------------+---------------+-------------------+-----------------+-----------+----+
|company      |ticker|country|revenue_usd_bn|scope1_emissions_mt_co2e|scope2_emissions_mt_co2e|scope3_emissions_mt_co2e|total_s1_s2_mt_co2e|yoy_scope1_change_pct|carbon_intensity_tco2e_per_musd|esg_score_0_100|cdp_score_encoded|net_zero_target_set_enc|sbti_committed_enc|emissions_disclosed_enc|third_party_verified_enc|commitment_score|esg_cdp_gap         |scope3_share      |log_carbon_intensity|credibility_gap|emission_trend_3y  |greenwashing_flag|sector     |year|
+-------------+------+-------+--------------+---------------

In [2]:
# =====================================================================
# ETAPE 2 — Split temporel + VectorAssembler + poids de classe
# =====================================================================
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler

TARGET = "greenwashing_flag"

FEATURE_COLS = [
    "scope1_emissions_mt_co2e", "scope2_emissions_mt_co2e",
    "scope3_emissions_mt_co2e", "total_s1_s2_mt_co2e",
    "yoy_scope1_change_pct", "carbon_intensity_tco2e_per_musd",
    "esg_score_0_100", "cdp_score_encoded",
    "net_zero_target_set_enc", "sbti_committed_enc",
    "emissions_disclosed_enc", "third_party_verified_enc",
    "commitment_score", "esg_cdp_gap", "scope3_share",
    "log_carbon_intensity", "credibility_gap", "emission_trend_3y",
]

# Split temporel strict (pas de split aleatoire — evite la fuite temporelle)
train_df = df.filter(F.col("year") <= 2020)
test_df  = df.filter(F.col("year") >  2020)

print(f"Train (year <= 2020) : {train_df.count()} lignes")
print(f"Test  (year >  2020) : {test_df.count()} lignes")

print("\nRepartition target TRAIN :")
train_df.groupBy(TARGET).count().orderBy(TARGET).show()
print("Repartition target TEST :")
test_df.groupBy(TARGET).count().orderBy(TARGET).show()

# Poids de classe — equivalent class_weight="balanced"
counts  = {r[TARGET]: r["count"] for r in train_df.groupBy(TARGET).count().collect()}
total   = sum(counts.values())
weights = {cls: total / (len(counts) * cnt) for cls, cnt in counts.items()}
print(f"Poids de classe : {weights}")

wmap     = F.create_map([F.lit(x) for pair in weights.items() for x in pair])
train_df = train_df.withColumn("classWeightCol", wmap[F.col(TARGET)])
test_df  = test_df.withColumn("classWeightCol", F.lit(1.0))

assembler       = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features", handleInvalid="skip")
train_assembled = assembler.transform(train_df).withColumnRenamed(TARGET, "label")
test_assembled  = assembler.transform(test_df).withColumnRenamed(TARGET, "label")

print(f"\nPret — train={train_assembled.count()}  test={test_assembled.count()}")

Train (year <= 2020) : 330 lignes


Test  (year >  2020) : 120 lignes

Repartition target TRAIN :


+-----------------+-----+
|greenwashing_flag|count|
+-----------------+-----+
|                0|  280|
|                1|   50|
+-----------------+-----+

Repartition target TEST :


+-----------------+-----+
|greenwashing_flag|count|
+-----------------+-----+
|                0|  105|
|                1|   15|
+-----------------+-----+



Poids de classe : {1: 3.3, 0: 0.5892857142857143}



Pret — train=330  test=120


In [3]:
# =====================================================================
# ETAPE 3 — Benchmark Spark MLlib
# Modeles : LogReg L1/L2/ElasticNet, RandomForest, DecisionTree, LinearSVC
# =====================================================================
import time
import pandas as pd

from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    DecisionTreeClassifier,
    LinearSVC,
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

ev_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
ev_pr  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
ev_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
ev_pre = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
ev_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

results       = []
fitted_models = {}  # stocke le modele entraine uniquement (jamais les DataFrames)

def run(name, model):
    print(f"\n{'='*55}\n  {name}\n{'='*55}")
    t0 = time.time()
    try:
        fitted = model.fit(train_assembled)
    except Exception as e:
        print(f"  ECHEC : {e}"); return
    elapsed = round(time.time() - t0, 1)

    preds = fitted.transform(test_assembled)
    try:
        roc = round(ev_roc.evaluate(preds), 4)
        pr  = round(ev_pr.evaluate(preds), 4)
    except Exception:
        roc, pr = None, None
    f1  = round(ev_f1.evaluate(preds), 4)
    pre = round(ev_pre.evaluate(preds), 4)
    rec = round(ev_rec.evaluate(preds), 4)

    print(f"  AUC-ROC={roc}  AUC-PR={pr}  F1={f1}  P={pre}  R={rec}  [{elapsed}s]")
    results.append({
        "model": name, "auc_roc": roc, "auc_pr": pr,
        "f1": f1, "precision": pre, "recall": rec, "time_s": elapsed
    })
    fitted_models[name] = fitted  # modele uniquement, pas les preds

# --- Regression Logistique ---
# elasticNetParam : 0.0 = L2 pure, 1.0 = L1 pure, 0.5 = ElasticNet
# weightCol = equivalent class_weight="balanced"
run("LogReg_L2", LogisticRegression(
    featuresCol="features", labelCol="label", weightCol="classWeightCol",
    regParam=0.1, elasticNetParam=0.0, maxIter=200, family="binomial"))

run("LogReg_L1", LogisticRegression(
    featuresCol="features", labelCol="label", weightCol="classWeightCol",
    regParam=0.1, elasticNetParam=1.0, maxIter=200, family="binomial"))

run("LogReg_ElasticNet", LogisticRegression(
    featuresCol="features", labelCol="label", weightCol="classWeightCol",
    regParam=0.1, elasticNetParam=0.5, maxIter=200, family="binomial"))

# --- Arbres ---
run("RandomForest", RandomForestClassifier(
    featuresCol="features", labelCol="label", weightCol="classWeightCol",
    numTrees=200, maxDepth=8, seed=42))

run("DecisionTree", DecisionTreeClassifier(
    featuresCol="features", labelCol="label", weightCol="classWeightCol",
    seed=42))

# --- LinearSVC (sans weightCol natif dans MLlib) ---
run("LinearSVC", LinearSVC(
    featuresCol="features", labelCol="label", maxIter=100))

# --- Recapitulatif ---
res_df = pd.DataFrame(results).sort_values("auc_roc", ascending=False).reset_index(drop=True)
print("\n" + "="*70)
print("BENCHMARK COMPLET — trie par AUC-ROC")
print("="*70)
print(res_df.to_string(index=False))

best_name = res_df.iloc[0]["model"]
print(f"\n>>> Meilleur modele : {best_name}  (AUC-ROC={res_df.iloc[0]['auc_roc']})")


  LogReg_L2


  AUC-ROC=0.8279  AUC-PR=0.299  F1=0.5921  P=0.901  R=0.525  [12.5s]

  LogReg_L1


  AUC-ROC=0.833  AUC-PR=0.3653  F1=0.6714  P=0.9052  R=0.6083  [7.7s]

  LogReg_ElasticNet


  AUC-ROC=0.8463  AUC-PR=0.3733  F1=0.5667  P=0.9  R=0.5  [7.1s]

  RandomForest


  AUC-ROC=0.9987  AUC-PR=0.9919  F1=0.9915  P=0.9917  R=0.9917  [9.8s]

  DecisionTree


  AUC-ROC=0.96  AUC-PR=0.9375  F1=0.9379  P=0.9495  R=0.9333  [5.8s]

  LinearSVC


  AUC-ROC=0.9175  AUC-PR=0.5235  F1=0.8883  P=0.9189  R=0.875  [37.2s]

BENCHMARK COMPLET — trie par AUC-ROC
            model  auc_roc  auc_pr     f1  precision  recall  time_s
     RandomForest   0.9987  0.9919 0.9915     0.9917  0.9917     9.8
     DecisionTree   0.9600  0.9375 0.9379     0.9495  0.9333     5.8
        LinearSVC   0.9175  0.5235 0.8883     0.9189  0.8750    37.2
LogReg_ElasticNet   0.8463  0.3733 0.5667     0.9000  0.5000     7.1
        LogReg_L1   0.8330  0.3653 0.6714     0.9052  0.6083     7.7
        LogReg_L2   0.8279  0.2990 0.5921     0.9010  0.5250    12.5

>>> Meilleur modele : RandomForest  (AUC-ROC=0.9987)


In [4]:
# =====================================================================
# ETAPE 4 — Feature importances + export livrables
# Les predictions sont recalculees ici depuis le modele stocke
# =====================================================================
import json
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

print(f"Modele retenu : {best_name}")

# Recalcul propre des predictions depuis test_assembled
final_model = fitted_models[best_name]
final_preds = final_model.transform(test_assembled)

# --- Metriques finales ---
metrics = {
    "model":     best_name,
    "auc_roc":   round(ev_roc.evaluate(final_preds), 4),
    "auc_pr":    round(ev_pr.evaluate(final_preds), 4),
    "f1_score":  round(ev_f1.evaluate(final_preds), 4),
    "precision": round(ev_pre.evaluate(final_preds), 4),
    "recall":    round(ev_rec.evaluate(final_preds), 4),
}
print("\nMetriques finales :")
for k, v in metrics.items():
    print(f"  {k}: {v}")

# --- Feature importances ---
fi_list = []
if hasattr(final_model, "featureImportances"):
    # RandomForest, DecisionTree
    arr     = final_model.featureImportances.toArray()
    fi_list = sorted(
        [{"feature": f, "importance": round(float(i), 6)}
         for f, i in zip(FEATURE_COLS, arr)],
        key=lambda x: x["importance"], reverse=True
    )
elif hasattr(final_model, "coefficients"):
    # LogisticRegression, LinearSVC
    arr     = final_model.coefficients.toArray()
    fi_list = sorted(
        [{"feature": f, "importance": round(abs(float(c)), 6)}
         for f, c in zip(FEATURE_COLS, arr)],
        key=lambda x: x["importance"], reverse=True
    )

print(f"\nTop 10 feature importances ({best_name}) :")
print(f"  {'Feature':<35} {'Importance':>10}  Barre")
print("  " + "-"*65)
for fi in fi_list[:10]:
    bar = "█" * int(fi["importance"] * 60)
    print(f"  {fi['feature']:<35} {fi['importance']:>10.4f}  {bar}")

# --- Export CSV predictions pour Zouin ---
# Colonnes : year | company | sector | greenwashing_flag | prediction | probability
preds_export = (
    final_preds
    .withColumn("prob_arr", vector_to_array(F.col("probability")))
    .select(
        F.col("year").cast("int"),
        F.col("company"),
        F.col("sector"),
        F.col("label").cast("int").alias("greenwashing_flag"),
        F.col("prediction").cast("int").alias("prediction"),
        F.round(F.col("prob_arr")[1], 4).alias("probability"),
    )
    .orderBy("company", "year")
)

print("\nApercu predictions :")
preds_export.show(10, truncate=False)

print("Confusion (greenwashing_flag vs prediction) :")
preds_export.groupBy("greenwashing_flag", "prediction").count() \
            .orderBy("greenwashing_flag", "prediction").show()

PRED_PATH = "s3a://sarangan/gold/ml/predictions/"
preds_export.coalesce(1).write.mode("overwrite").option("header", "true").csv(PRED_PATH)
print(f"Predictions exportees : {PRED_PATH}")

# --- Rapport JSON ---
report = {
    "model_name":          best_name,
    "metrics":             metrics,
    "feature_importances": fi_list,
    "train_period":        "year <= 2020",
    "test_period":         "year > 2020",
    "n_train":             330,
    "n_test":              120,
    "class_weights":       {str(k): round(v, 4) for k, v in weights.items()},
    "note": (
        "Split temporel strict — pas de data leakage. "
        "Desequilibre 85/15 compense par weightCol. "
        "GBT exclu (OOMKill cluster Onyxia). "
        "Test set = 120 obs sur 4 ans."
    ),
    "all_models_benchmark": res_df.to_dict(orient="records"),
}

LOCAL_JSON = "/tmp/rapport_modele_ds.json"
with open(LOCAL_JSON, "w") as f:
    json.dump(report, f, indent=2, default=str)
print(f"JSON sauvegarde localement : {LOCAL_JSON}")

try:
    import boto3
    s3 = boto3.client("s3", endpoint_url="https://minio.lab.sspcloud.fr")
    s3.upload_file(LOCAL_JSON, "sarangan", "gold/dashboard/rapport_modele_ds.json")
    print("JSON uploade : s3a://sarangan/gold/dashboard/rapport_modele_ds.json")
except Exception as e:
    print(f"Upload JSON : {e}")
    print(f"Fichier disponible localement : {LOCAL_JSON}")

print("\n" + "="*65)
print("LIVRABLES PRETS")
print("="*65)
print(f"1. Predictions CSV : {PRED_PATH}")
print(f"2. JSON metriques  : s3a://sarangan/gold/dashboard/rapport_modele_ds.json")
print(f"   Colonnes CSV    : year | company | sector | greenwashing_flag | prediction | probability")

Modele retenu : RandomForest



Metriques finales :
  model: RandomForest
  auc_roc: 0.9987
  auc_pr: 0.9919
  f1_score: 0.9915
  precision: 0.9917
  recall: 0.9917

Top 10 feature importances (RandomForest) :
  Feature                             Importance  Barre
  -----------------------------------------------------------------
  yoy_scope1_change_pct                   0.2796  ████████████████
  esg_score_0_100                         0.2367  ██████████████
  cdp_score_encoded                       0.0649  ███
  scope1_emissions_mt_co2e                0.0567  ███
  total_s1_s2_mt_co2e                     0.0522  ███
  emission_trend_3y                       0.0451  ██
  esg_cdp_gap                             0.0439  ██
  scope2_emissions_mt_co2e                0.0436  ██
  scope3_emissions_mt_co2e                0.0404  ██
  carbon_intensity_tco2e_per_musd         0.0379  ██



Apercu predictions :


+----+-----------------------+-----------+-----------------+----------+-----------+
|year|company                |sector     |greenwashing_flag|prediction|probability|
+----+-----------------------+-----------+-----------------+----------+-----------+
|2021|3M                     |Industrials|0                |0         |0.327      |
|2022|3M                     |Industrials|0                |0         |0.2481     |
|2023|3M                     |Industrials|0                |0         |0.3328     |
|2024|3M                     |Industrials|0                |0         |0.0282     |
|2021|American Electric Power|Utilities  |1                |1         |0.8826     |
|2022|American Electric Power|Utilities  |0                |0         |0.0579     |
|2023|American Electric Power|Utilities  |0                |0         |0.0668     |
|2024|American Electric Power|Utilities  |0                |0         |0.0571     |
|2021|ArcelorMittal          |Industrials|0                |0         |0.105

+-----------------+----------+-----+
|greenwashing_flag|prediction|count|
+-----------------+----------+-----+
|                0|         0|  105|
|                1|         0|    1|
|                1|         1|   14|
+-----------------+----------+-----+



Predictions exportees : s3a://sarangan/gold/ml/predictions/
JSON sauvegarde localement : /tmp/rapport_modele_ds.json
Upload JSON : No module named 'boto3'
Fichier disponible localement : /tmp/rapport_modele_ds.json

LIVRABLES PRETS
1. Predictions CSV : s3a://sarangan/gold/ml/predictions/
2. JSON metriques  : s3a://sarangan/gold/dashboard/rapport_modele_ds.json
   Colonnes CSV    : year | company | sector | greenwashing_flag | prediction | probability
